# 05 - Trajectories: emotion dynamics across a story's tokens (Q3)

**What this notebook is for.** The third research question: in stories that move through
three emotions (sequentially or simultaneously), how does the per-token cosine between the
residual stream and each emotion's probe direction evolve — gradual ramp-and-crossover, or
steps at lexical cues? This notebook exhibits the collected substrate and the figure
vocabulary. **It draws no verdicts**: the scoring reads for Q3.H1.E1 (ramp-vs-step statistic,
anticipation window, layer contrast, pass bars) are not yet registered in TREE.md, and per the
anti-peeking discipline the full-corpus aggregate stays out of this notebook until they are.
Everything here is single-story exhibition plus descriptive corpus statistics.

**Data lineage (read this first).**
- **Stories**: 5,888 three-emotion stories generated by `google/gemma-4-31b-it` *itself*
  (the self-generated convention, TREE Q1.H2.E10) — 173 emotion triples x 6 permutations x
  2 modes x 3 samples, seed 20260721, temperature 1.3. QC dropped emotion-word leaks (189)
  and tag mismatches (151). HF: `abotresol/emotion-combined-stories-gemma-4-31b-it`.
- **Trajectories**: teacher-forced forward pass through the same model (bf16, right padding —
  post-dates the padding-side instrument fix, TREE Q1.H3.E4), residual captured at layers
  {6, 15, 24, 33, 42, 51}. Each story's shard stores raw per-token dots onto 207 unit probes:
  171 **corpus-lineage** (external 4B-generated stories), 12 **selfgen-lineage** (n=256
  self-generated, the E10 winner — only the 12 battery emotions have this lineage), and 24
  fixed-seed **random** null directions. The two probe lineages are measurably different
  objects (contrast cosine 0.22, TREE Q1.H2.E6) — figures state which one they use.
  HF: `abotresol/emotion-combined-trajectories-gemma-4-31b-it`.
- **Convention**: all cosines below are **centered** (per-story token-mean removed — the E9
  lesson that uncentered readouts bury affect under shared story-reading structure), computed
  from the shards' raw dots and stored centered norms; smoothing windows are stated per figure.


In [1]:
import json

import numpy as np

from emotion_vectors.artifacts import fetch
from emotion_vectors import trajectory_plots as tp
from emotion_vectors.trajectories import transition_windows

manifest = [json.loads(l) for l in fetch("combined_trajectories/manifest.jsonl").read_text().splitlines()]
labels = json.loads(fetch("combined_trajectories/probe_labels.json").read_text())
config = json.loads(fetch("combined_trajectories/run_config.json").read_text())
LAYERS = config["layers"]
print(f"{len(manifest)} stories | layers {LAYERS} | {len(labels)} probes")


5888 stories | layers [6, 15, 24, 33, 42, 51] | 207 probes


## 1. What was collected

Descriptive statistics only — the corpus as measured, before any hypothesis touches it.

In [2]:
from collections import Counter

modes = Counter(m["mode"] for m in manifest)
tokens = np.array([m["n_tokens"] for m in manifest])
cats = Counter(m["category"] for m in manifest)
print(f"modes: {dict(modes)}")
print(f"tokens/story: median {np.median(tokens):.0f}, range {tokens.min()}-{tokens.max()}")
print(f"categories: {dict(cats)}")
seq3 = sum(1 for m in manifest if m["mode"] == "SEQUENTIAL" and len(m["phase_token_starts"]) == 3)
print(f"sequential stories with clean 3-phase alignment: {seq3}/{modes['SEQUENTIAL']}")


modes: {'SIMULTANEOUS': 2878, 'SEQUENTIAL': 3010}
tokens/story: median 210, range 171-365
categories: {'B_conflict': 1644, 'E_arousal_mismatch': 1092, 'F_valence_spread': 1084, 'A_superposition': 993, 'D_timescale': 1075}
sequential stories with clean 3-phase alignment: 3009/3010


## 2. One sequential story, every view

Story `t000_seq_p2_2f9faf62` (upset -> unsettled -> cheerful, 236 tokens) — chosen as the
first mid-length cross-valence sequential story in the manifest, before any scoring existed.
Corpus-lineage probes (the triple's emotions are not all in the 12-emotion selfgen set).
Each figure carries its own how-to-read caption.

In [3]:
STORY = "t000_seq_p2_2f9faf62"
row = next(m for m in manifest if m["story_id"] == STORY)
shard = np.load(fetch(f"combined_trajectories/shards/{STORY}.npz"))
emotions, starts = row["phase_emotions"], row["phase_token_starts"]
L33 = LAYERS.index(33)
cos33 = tp.smooth(tp.story_cosines(shard, labels, emotions, L33, lineage="corpus"), window=8)
tp.lines_figure(cos33, emotions, starts).show()


In [4]:
tp.ternary_figure(cos33, emotions, starts).show()

In [5]:
per_layer = [
    tp.smooth(tp.story_cosines(shard, labels, emotions, LAYERS.index(k), lineage="corpus"), window=8)
    for k in (6, 33, 51)
]
tp.layer_ternaries(per_layer, [f"layer {k}" for k in (6, 33, 51)], emotions).show()


In [6]:
tp.speed_figure(shard["speed"].astype(np.float32)[:, L33], starts).show()

**The confound check** — all 12 selfgen-lineage probes (plus the triple's corpus probes)
over the same story. The assigned emotion's row should dominate its own phase; any off-triple
row hot everywhere would mean the probes read valence/style, not emotion identity.

In [7]:
battery_idx = [i for i, name in enumerate(labels) if name.startswith("selfgen:")]
triple_idx = [labels.index(f"corpus:{e}") for e in emotions]
rows_idx = battery_idx + triple_idx
row_names = [labels[i] for i in rows_idx]
d = shard["dots"].astype(np.float32)[:, L33]
d = d - d.mean(0, keepdims=True)
heat = d[:, rows_idx] / np.clip(shard["norms_centered"].astype(np.float32)[:, L33][:, None], 1e-6, None)
tp.probe_heatmap_figure(heat, row_names, starts).show()


## 3. One simultaneous story, for contrast

A SIMULTANEOUS story should show all three emotions co-active — in the ternary view, a
trajectory hovering near the centroid rather than touring the corners.

In [8]:
sim = next(m for m in manifest if m["mode"] == "SIMULTANEOUS" and 150 < m["n_tokens"] < 260)
sshard = np.load(fetch(f"combined_trajectories/shards/{sim['story_id']}.npz"))
scos = tp.smooth(tp.story_cosines(sshard, labels, sim["emotions"], L33, lineage="corpus"), window=8)
print(sim["story_id"], sim["emotions"])
tp.ternary_figure(scos, sim["emotions"], [0]).show()


t000_sim_p0_5bb39df3 ['unsettled', 'upset', 'cheerful']


## 4. Registered analysis — pending

The population verdict comes from the transition-locked average (all sequential transitions
aligned at t=0, incoming vs outgoing curves, bootstrap bands, per layer band) scored against
the random-direction null — `transition_windows` and `transition_locked_figure` are built and
tested, but the reads (ramp-vs-step statistic, anticipation window, pass bars) must be
registered in TREE.md **before** that figure is rendered over the corpus. This section will
hold the registered analysis and its verdict; until then, deliberately empty.